# Kaggle — House Prices : Advanced Regression Techniques

In [285]:
# Recharge automatiquement src/transformers.py quand il est modifié
%load_ext autoreload
%autoreload 2

# Rend le dossier src/ (un niveau au-dessus de notebooks/) importable
import sys
sys.path.append("..")
from src.transformers import GroupMedianImputer, ModeImputer

import pandas as pd
import seaborn as sns
import numpy as np

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  # Id du test, nécessaires pour la soumission

train.shape, test.shape

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


((1460, 81), (1459, 80))

## 1. Préparation des données

In [286]:
train = train.reset_index(drop=True)

# Cible passée en log (la métrique Kaggle est un RMSE sur le log du prix)
y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)

In [287]:
all_X.shape  # attendu : (2919, 79)

(2919, 79)

## 2. Valeurs manquantes

In [288]:
# Colonnes qui contiennent au moins un NaN
cols_na = all_X.columns[all_X.isnull().any()]
df_null = all_X[cols_na]

# Nombre et pourcentage de NaN, triés par ordre décroissant
na = df_null.isnull().sum().sort_values(ascending=False)
na_pct = (na / len(all_X) * 100).round(1)
pd.DataFrame({"nb_NaN": na, "pct": na_pct})

,nb_NaN,pct
PoolQC,2909,99.7
MiscFeature,2814,96.4
Alley,2721,93.2
Fence,2348,80.4
MasVnrType,1766,60.5
FireplaceQu,1420,48.6
LotFrontage,486,16.6
GarageQual,159,5.4
GarageYrBlt,159,5.4
GarageCond,159,5.4


### 2.1 `PoolQC` — qualité de la piscine (99,7 % de NaN)

D'après `data_description.txt`, `NA` = pas de piscine. On vérifie que les NaN correspondent aux maisons avec `PoolArea == 0`.

In [289]:
# Seulement 10 piscines ont une qualité renseignée
all_X['PoolQC'].value_counts()

PoolQC
Ex    4
Gd    4
Fa    2
Name: count, dtype: int64

**Test de cohérence `PoolQC` × `PoolArea`**

In [290]:
# Maisons sans piscine
(all_X['PoolArea'] == 0).sum()

np.int64(2906)

In [291]:
# Pas de piscine ET pas de qualité -> cohérent
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(2906)

In [292]:
# Piscine présente MAIS qualité manquante -> vraies valeurs manquantes
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(3)

In [293]:
# Pas de piscine MAIS qualité renseignée -> incohérence
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(0)

In [294]:
# Piscine présente ET qualité renseignée -> cohérent
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(10)

**Résultat**

| | `PoolQC` vide | `PoolQC` renseigné |
|---|---|---|
| `PoolArea == 0` | 2 906 | 0 |
| `PoolArea > 0` | 3 | 10 |

- 2 906 NaN = pas de piscine.
- 3 piscines sans qualité = vraies valeurs manquantes.
- Aucune incohérence dans l'autre sens.

In [295]:
# Les 3 piscines SANS qualité : train ou test ?
mask = (all_X["PoolArea"] > 0) & (all_X["PoolQC"].isnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([2420, 2503, 2599], dtype='int64')
['test', 'test', 'test']


In [296]:
# Les 10 piscines AVEC qualité : train ou test ?
mask = (all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([197, 810, 1170, 1182, 1298, 1386, 1423, 1974, 2573, 2710], dtype='int64')
['train', 'train', 'train', 'train', 'train', 'train', 'train', 'test', 'test', 'test']


**Conclusion**

- Les 3 piscines sans qualité sont toutes dans le test.
- Seulement 7 piscines notées dans le train sur 1 460 maisons : trop peu pour que le modèle apprenne l'effet de la qualité.
- `PoolArea` contient déjà l'information sur la présence de piscine.

**Décision : suppression de `PoolQC`.**

In [297]:
# Suppression de PoolQC (trop peu d'exemples notés dans le train)
all_X = all_X.drop(columns=["PoolQC"])
all_X.shape  # attendu : (2919, 78)

(2919, 78)

### 2.2 `MiscFeature` — équipement divers (96,4 % de NaN)

D'après `data_description.txt`, `NA` = aucun équipement, et `MiscVal` = valeur en $ de cet équipement. On vérifie que les NaN correspondent aux maisons avec `MiscVal == 0`.

In [298]:
# Répartition des équipements renseignés
all_X['MiscFeature'].value_counts()

MiscFeature
Shed    95
Gar2     5
Othr     4
TenC     1
Name: count, dtype: int64

In [299]:
# Nombre total d'équipements renseignés
all_X['MiscFeature'].value_counts().sum()

np.int64(105)

**Test de cohérence `MiscFeature` × `MiscVal`**

In [300]:
# Maisons sans valeur d'équipement
(all_X['MiscVal'] == 0).sum()

np.int64(2816)

In [301]:
# Pas de valeur ET pas d'équipement -> cohérent
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(2813)

In [302]:
# Valeur renseignée MAIS équipement manquant -> incohérence
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(1)

In [303]:
# Équipement présent MAIS valeur à 0
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(3)

In [304]:
# Équipement présent ET valeur renseignée -> cohérent
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(102)

**Résultat**

| | `MiscFeature` vide | `MiscFeature` renseigné |
|---|---|---|
| `MiscVal == 0` | 2 813 | 3 |
| `MiscVal > 0` | 1 | 102 |

- 2 813 NaN = aucun équipement.
- 1 maison avec une valeur mais sans équipement, 3 équipements avec une valeur à 0.

In [305]:
# Répartition des équipements par catégorie, entre train et test
condition = (all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())
sub = all_X.loc[condition, ['MiscFeature']].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MiscFeature'], sub['origine'], margins=True)

origine,test,train,All
MiscFeature,,,
Gar2,3,2,5
Othr,2,1,3
Shed,45,48,93
TenC,0,1,1
All,50,52,102


**Conclusion**

- Répartition train / test d'environ 50/50 (52 / 50).
- Dans le train, seule `Shed` a assez d'exemples (48) ; `Gar2`, `Othr` et `TenC` en ont 1 ou 2 : le modèle ne peut rien en apprendre.
- `MiscVal` porte déjà l'information utile (présence d'un équipement et sa valeur) ; la catégorie apporte très peu en plus.
- Garder `MiscFeature` imposerait un encodage pour un gain quasi nul.

**Décision : suppression de `MiscFeature`, on conserve `MiscVal`.**

In [306]:
# Suppression de MiscFeature (MiscVal conserve l'information utile)
all_X = all_X.drop(columns=["MiscFeature"])
all_X.shape  # attendu : (2919, 77)

(2919, 77)

### 2.3 `Alley` — type d'accès par allée (93,2 % de NaN)

D'après `data_description.txt`, `NA` = pas d'accès par une allée. Aucune autre colonne ne décrit l'allée : pas de test de cohérence possible, le NaN se lit directement comme « absent ».

In [307]:
# Répartition des types d'allée renseignés
all_X['Alley'].value_counts()

Alley
Grvl    120
Pave     78
Name: count, dtype: int64

In [308]:
# Nombre total d'allées renseignées
all_X['Alley'].value_counts().sum()

np.int64(198)

In [309]:
# Répartition des types d'allée entre train et test
sub = all_X[["Alley"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['Alley'], sub['origine'], margins=True)

origine,test,train,All
Alley,,,
Grvl,70,50,120
Pave,37,41,78
All,107,91,198


**Conclusion**

- Effectifs suffisants dans le train : `Grvl` 50, `Pave` 41.
- Pas d'ordre entre gravier et pavé : encodage one-hot (pas ordinal).
- Aucune autre colonne ne porte cette information.

**Décision : NaN remplacés par `"None"`.**
Le one-hot sera fait à la fin, en une seule fois sur toutes les colonnes catégorielles ; la colonne `Alley_None` pourra alors être supprimée (redondante avec `Alley_Grvl = Alley_Pave = 0`).

In [310]:
# NaN = pas d'allée
all_X["Alley"] = all_X["Alley"].fillna("None")

In [311]:
# Vérification : plus aucun NaN dans Alley
all_X['Alley'].value_counts()

Alley
None    2721
Grvl     120
Pave      78
Name: count, dtype: int64

### 2.4 `Fence` — qualité de la clôture (80,4 % de NaN)

D'après `data_description.txt`, `NA` = pas de clôture. Valeurs : `GdPrv` (bonne intimité), `MnPrv` (intimité minimale), `GdWo` (bon bois), `MnWw` (bois/grillage minimal). Aucune autre colonne ne décrit la clôture : le NaN se lit directement comme « absent ».

In [312]:
# Répartition des types de clôture renseignés
all_X['Fence'].value_counts()

Fence
MnPrv    329
GdPrv    118
GdWo     112
MnWw      12
Name: count, dtype: int64

In [313]:
# Nombre total de clôtures renseignées
all_X['Fence'].value_counts().sum()

np.int64(571)

In [314]:
# Répartition des types de clôture entre train et test
sub = all_X[["Fence"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['Fence'], sub['origine'], margins=True)

origine,test,train,All
Fence,,,
GdPrv,59,59,118
GdWo,58,54,112
MnPrv,172,157,329
MnWw,1,11,12
All,290,281,571


**Conclusion**

- Répartition train / test d'environ 50/50 (281 / 290).
- Effectifs suffisants dans le train sauf `MnWw` (11).
- Deux familles de clôture (intimité / bois), chacune avec un niveau minimum et un niveau bon.

**Décision : NaN remplacés par `"None"`.**
Encodage à l'étape finale. Piste à comparer en CV avec un one-hot classique : deux colonnes ordinales `Fence_Privacy` (None 0, MnPrv 1, GdPrv 2) et `Fence_Wood` (None 0, MnWw 1, GdWo 2), possibles car chaque maison a au plus une clôture.

In [315]:
# NaN = pas de clôture, puis vérification
all_X["Fence"] = all_X["Fence"].fillna("None")
all_X['Fence'].value_counts()

Fence
None     2348
MnPrv     329
GdPrv     118
GdWo      112
MnWw       12
Name: count, dtype: int64

### 2.5 `MasVnrType` — type de placage maçonnerie (60,5 % de NaN)

D'après `data_description.txt`, `MasVnrType` n'a pas de valeur `NA` mais une valeur `None` (pas de placage) : pandas convertit automatiquement le texte `"None"` en NaN à la lecture. `MasVnrArea` (surface du placage) a aussi 23 NaN. On croise les deux colonnes.

In [316]:
# Répartition des types de placage renseignés
all_X['MasVnrType'].value_counts()

MasVnrType
BrkFace    879
Stone      249
BrkCmn      25
Name: count, dtype: int64

In [317]:
# Croise 'colonne catégorielle vide / renseignée' avec 'colonne numérique NaN / 0 / > 0'
# (réutilisable pour les groupes Garage*, Bsmt*, Fireplace*...)
def coherence(df, col_cat, col_num):
    etat_cat = df[col_cat].isnull().map({True: "vide", False: "renseigné"}).rename(col_cat)
    etat_num = pd.Series(
        np.select([df[col_num].isnull(), df[col_num] == 0], ["NaN", "0"], default="> 0"),
        index=df.index, name=col_num
    )
    return pd.crosstab(etat_cat, etat_num, margins=True)

In [318]:
# Test de cohérence MasVnrType × MasVnrArea
coherence(all_X, "MasVnrType", "MasVnrArea")

MasVnrArea,0,> 0,NaN,All
MasVnrType,,,,
renseigné,3,1150,0,1153
vide,1735,8,23,1766
All,1738,1158,23,2919


In [319]:
# Répartition des types de placage entre train et test
sub = all_X[["MasVnrType"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MasVnrType'], sub['origine'], margins=True)

origine,test,train,All
MasVnrType,,,
BrkCmn,10,15,25
BrkFace,434,445,879
Stone,121,128,249
All,565,588,1153


**Conclusion**

- 1 735 maisons sans type et sans surface : pas de placage.
- 23 maisons sans type ni surface : vraies valeurs manquantes sur les deux colonnes ; la majorité des maisons n'ayant pas de placage, on les considère sans placage.
- 8 maisons avec une surface mais sans type : la surface prouve qu'il y a un placage. On leur attribuera le type le plus fréquent **parmi les maisons qui ont un placage** ; ce mode est une statistique apprise, il sera donc calculé dans le **Pipeline** (sur la partie entraînement de chaque fold) pour éviter toute fuite.
- 3 maisons avec un type mais une surface à 0 : laissées telles quelles (pas de NaN, impossible de savoir quelle colonne est fausse).
- Pas d'ordre entre les types ; `BrkCmn` a 15 exemples dans le train, suffisant pour le garder.

**Décision** : `MasVnrType` → `"None"` pour les maisons sans placage, `MasVnrArea` → 0. Les 8 maisons restent NaN et seront imputées dans le Pipeline. One-hot à l'étape finale.

In [320]:
# Pas de type ET (surface à 0 ou vide) -> pas de placage
masque = ( ((all_X["MasVnrArea"] == 0) |(all_X["MasVnrArea"].isna()) )  & (all_X["MasVnrType"].isna() )) 

all_X.loc[masque, "MasVnrType"] = "None"
# Les 8 maisons avec une surface > 0 restent NaN : imputées dans le Pipeline

In [321]:
# Surface manquante = pas de placage
all_X["MasVnrArea"] = all_X["MasVnrArea"].fillna(0)

In [322]:
# Vérification : None uniquement avec une surface à 0, 8 NaN restants (surface > 0)
etat_area = np.where(all_X["MasVnrArea"] > 0, "> 0", "0")
pd.crosstab(all_X["MasVnrType"], etat_area, margins=True, colnames=["MasVnrArea"], dropna=False)

MasVnrArea,0,> 0,All
MasVnrType,,,
BrkCmn,0,25,25
BrkFace,2,877,879
None,1758,0,1758
Stone,1,248,249
NaN,0,8,8
All,1761,1158,2919


### 2.6 `FireplaceQu` — qualité de la cheminée (48,6 % de NaN)

D'après `data_description.txt`, `NA` = pas de cheminée. Échelle de qualité : `Po` < `Fa` < `TA` < `Gd` < `Ex`. On vérifie la cohérence avec `Fireplaces` (nombre de cheminées).

In [323]:
# Répartition des niveaux de qualité renseignés
all_X['FireplaceQu'].value_counts()

FireplaceQu
Gd    744
TA    592
Fa     74
Po     46
Ex     43
Name: count, dtype: int64

In [324]:
# Nombre total de cheminées avec une qualité renseignée
all_X['FireplaceQu'].value_counts().sum()

np.int64(1499)

In [325]:
# Test de cohérence FireplaceQu × Fireplaces
coherence(all_X, "FireplaceQu", "Fireplaces")

Fireplaces,0,> 0,All
FireplaceQu,,,
renseigné,0,1499,1499
vide,1420,0,1420
All,1420,1499,2919


**Conclusion**

- Cohérence parfaite : les 1 420 NaN correspondent exactement aux maisons sans cheminée, les 1 499 qualités renseignées aux maisons avec au moins une cheminée.
- Échelle de qualité ordonnée : encodage **ordinal** avec le dictionnaire commun des qualités (`None` 0, `Po` 1, `Fa` 2, `TA` 3, `Gd` 4, `Ex` 5). Les niveaux rares (`Po`, `Ex`) ne posent pas de problème en ordinal.

**Décision : NaN remplacés par `"None"`.** Encodage ordinal à l'étape finale.

In [326]:
# NaN = pas de cheminée, puis vérification
all_X["FireplaceQu"] = all_X["FireplaceQu"].fillna("None")
all_X['FireplaceQu'].value_counts()

FireplaceQu
None    1420
Gd       744
TA       592
Fa        74
Po        46
Ex        43
Name: count, dtype: int64

### 2.7 `LotFrontage` — longueur de façade sur rue (16,6 % de NaN)

Chaque terrain a une façade : les NaN sont de **vraies valeurs manquantes**, pas des « absents ». Il faut donc les imputer.

In [327]:
# Distribution sur le train
all_X.iloc[:n_train]['LotFrontage'].describe()

count    1201.000000
mean       70.049958
std        24.284752
min        21.000000
25%        59.000000
50%        69.000000
75%        80.000000
max       313.000000
Name: LotFrontage, dtype: float64

In [328]:
# Médiane globale sur le train
all_X.iloc[:n_train]['LotFrontage'].median()

np.float64(69.0)

In [329]:
# Médiane et effectif par quartier sur le train
all_X.iloc[:n_train].groupby("Neighborhood")["LotFrontage"].agg(["median", "count"])

,median,count
Neighborhood,,
Blmngtn,43.0,14
Blueste,24.0,2
BrDale,21.0,16
BrkSide,52.0,51
ClearCr,80.0,13
CollgCr,70.0,126
Crawfor,74.0,41
Edwards,65.5,92
Gilbert,65.0,49


**Conclusion de l'analyse**

- Moyenne (70,0) et médiane (69) proches sur le train ; la médiane reste préférée, plus robuste aux valeurs extrêmes (max 313).
- La façade varie fortement selon le quartier : médianes de 21 (`BrDale`, `MeadowV`) à 91 (`NoRidge`). Une médiane globale serait très imprécise.
- Certains quartiers ont très peu de terrains dans le train (`Blueste` : 2) : une médiane globale de secours est nécessaire pour un quartier absent des données d'apprentissage.

**Décision : imputation par la médiane du quartier, médiane globale en secours.**
Cette médiane est une statistique apprise : elle est calculée dans un transformer scikit-learn (`GroupMedianImputer`, dans `src/transformers.py`) intégré au **Pipeline**, pour être réapprise sur la partie entraînement de chaque fold de la CV, sans fuite. Les NaN restent donc dans `all_X`.

In [330]:
# Test du transformer : fit sur le train uniquement
imp = GroupMedianImputer()
imp.fit(all_X.iloc[:n_train])

print(imp.medians_)        # doit correspondre au groupby ci-dessus
print(imp.global_median_)  # doit valoir 69.0

Neighborhood
Blmngtn    43.0
Blueste    24.0
BrDale     21.0
BrkSide    52.0
ClearCr    80.0
CollgCr    70.0
Crawfor    74.0
Edwards    65.5
Gilbert    65.0
IDOTRR     60.0
MeadowV    21.0
Mitchel    73.0
NAmes      73.0
NPkVill    24.0
NWAmes     80.0
NoRidge    91.0
NridgHt    88.5
OldTown    60.0
SWISU      60.0
Sawyer     71.0
SawyerW    66.5
Somerst    73.5
StoneBr    61.5
Timber     85.0
Veenker    68.0
Name: LotFrontage, dtype: float64
69.0


In [331]:
# Test du transformer : transform sur tout all_X
res = imp.transform(all_X)
print(res["LotFrontage"].isna().sum())     # attendu : 0 (tout est rempli)
print(all_X["LotFrontage"].isna().sum())   # attendu : 486 (all_X d'origine intact)

0
486


### 2.8 `MasVnrType` — imputation des 8 maisons avec surface mais sans type

Après le nettoyage de la section 2.5, les seuls NaN restants de `MasVnrType` sont les 8 maisons avec `MasVnrArea > 0`. Leur type est imputé par le type le plus fréquent **parmi les maisons qui ont un placage**.

Le mode de toute la colonne ne convient pas : `"None"` est désormais une catégorie (1 758 maisons), elle serait la plus fréquente. Le transformer `ModeImputer` (dans `src/transformers.py`) exclut donc cette valeur avant de calculer le mode, et sera intégré au Pipeline.

In [332]:
# Test du transformer : fit sur le train uniquement
mode_imp = ModeImputer()
mode_imp.fit(all_X.iloc[:n_train])
print(mode_imp.mode_)    # attendu : BrkFace

BrkFace


In [333]:
# Test du transformer : transform sur tout all_X
res = mode_imp.transform(all_X)
print(res["MasVnrType"].isna().sum())       # attendu : 0
print(res["MasVnrType"].value_counts())     # BrkFace : 879 -> 887
print(all_X["MasVnrType"].isna().sum())     # attendu : 8 (all_X d'origine intact)

0
MasVnrType
None       1758
BrkFace     887
Stone       249
BrkCmn       25
Name: count, dtype: int64
8


**Conclusion**

- Le mode appris sur le train (hors `"None"`) est `BrkFace` ; les 8 maisons passent en `BrkFace` (879 → 887).
- Plus aucun NaN dans `MasVnrType` après le transformer, et `all_X` d'origine reste intact.
- Hypothèse du transformer : le nettoyage de la section 2.5 est fait avant lui (seules les maisons à imputer sont encore NaN).